# V2_2 – Back up definitions and data to ADLS (secondary copy)

Run this **after the change freeze starts and before `V2_3_Git_Snapshot`**. The workspace's own Git link
(V2_1) is the primary copy of the definitions; this notebook saves the **secondary copy of the definitions and
all the data** in ADLS. V2_3 checks this run, V2_4 refuses to delete an item whose backup isn't OK, and V2_6
loads the data back after the move.

| Item type | Saved in ADLS | Restored by |
|---|---|---|
| Notebook, DataPipeline, Dataflow, DataAgent, GraphModel, Ontology | definition | V2_5 (Git); this copy is the fallback |
| Lakehouse | definition, `Files/` and `Tables/` (Delta, including every schema of a schema-enabled Lakehouse; the layout is saved in `lakehouse.json`). Shortcuts: definition only | V2_6 |
| Warehouse | definition (SQL project), tables as Delta folders | V2_6 |
| Eventhouse / KQL database | definition, schema script, and the rows as chosen in `KQL_BACKUP_MODE` | V2_6 |
| SQLEndpoint | views, functions, procedures and permissions as scripts | V2_6 |

### Eventhouse data: `KQL_BACKUP_MODE`
| Mode | Where the rows go | Restore (V2_6) | Needs |
|---|---|---|---|
| `"parquet"` (default) | Parquet files in ADLS, exported by the Eventhouse | `.ingest` from ADLS | Runner: Storage Blob Data Contributor (the Eventhouse writes with your identity) |
| `"adx"` | An Azure Data Explorer cluster, with Kusto Copy (reruns copy only new rows) | `.set-or-append` from ADX into the new Eventhouse | ADX cluster and databases; service principal (step 7); runner: Viewer on the ADX databases |
| `"both"` | Both | Parquet first, ADX as fallback | Both |

### Before the first run
1. **Python notebook** (not PySpark), in a workspace that is **not** being migrated. Run it as a person with
   **Contributor or higher** on every listed workspace.
2. **ADLS Gen2**: same storage account and container as V2_3–V2_6. Runner: **Storage Blob Data Contributor**.
3. Set `FREEZE_STARTED_UTC` (when nobody changes items any more). The run refuses to start before it.
4. Keep `DRY_RUN = True` first: it checks access and reports sizes without writing. Then `DRY_RUN = False`.
5. At the end the notebook prints the settings to paste into V2_3 (`SECONDARY_BACKUP_RUN_ID`, `FREEZE_STARTED_UTC`).

### What isn't backed up
* Data behind shortcuts (only the shortcut definition), data in Dataflow staging.
* KQL materialized views' stored results (their definitions are in the schema script), and rows ingested while
  the export runs (that's why the freeze comes first).
* Item permissions, workspace roles, connections and gateway credentials, sensitivity labels.

In [ ]:
# ---------------------------------------------------------------- what to back up
# Workspace IDs (GUIDs) to back up: the same workspaces as V2_1 and V2_3.
WORKSPACE_IDS = [
    # "00000000-0000-0000-0000-000000000000",
]
ITEM_TYPES = ["Dataflow", "DataAgent", "Eventhouse", "GraphModel", "Lakehouse", "Ontology",
              "SQLEndpoint", "Warehouse", "DataPipeline", "Notebook"]
INCLUDE_KQL_DATABASES = True        # an Eventhouse's data lives in its KQL databases: keep True
ONLY_ITEM_IDS = []                  # optional: only these item IDs

# When the change freeze started (UTC, e.g. "2026-11-16T10:00:00Z"). This run must start after it; V2_3 checks the
# same value. Leave empty only for a test run (V2_3 will then block).
FREEZE_STARTED_UTC = ""

# True = report what would be backed up and how big it is, without writing anything.
DRY_RUN = True

# ---------------------------------------------------------------- ADLS Gen2 (same account as V2_3-V2_6)
ADLS_ACCOUNT = "<storage-account-name>"   # hierarchical namespace enabled
ADLS_CONTAINER = "fabric-backup"          # must already exist
ADLS_ROOT = "fabric-backups"              # each run writes to <ADLS_ROOT>/<run id>/
KEY_VAULT_URL = "https://<key-vault-name>.vault.azure.net/"

COPY_LAKEHOUSE_FILES = True         # Lakehouse Files/ (shortcuts are skipped, not copied)
COPY_LAKEHOUSE_TABLES = True        # Lakehouse Tables/ as Delta folders (shortcuts skipped)
COPY_WAREHOUSE_TABLES = True        # Warehouse tables as Delta folders, read from OneLake
SCRIPT_SQL_ITEM_TYPES = ["SQLEndpoint"]  # script views/functions/procedures + permissions
MAX_PARALLEL_FILES = 8              # concurrent file copies per item

# ---------------------------------------------------------------- Eventhouse (KQL database) data
# "parquet" = every table exported to Parquet files in ADLS; V2_6 ingests them into the new Eventhouse.
# "adx"     = every table copied into an Azure Data Explorer cluster with Kusto Copy; V2_6 copies them back
#             from ADX into the new Eventhouse. Needs the ADX settings below.
# "both"    = both copies (safest; V2_6 uses the Parquet files and falls back to ADX).
# The KQL schema script (database_schema.kql) is saved in ADLS in every mode.
KQL_BACKUP_MODE = "parquet"

# ---------------------------------------------------------------- Azure Data Explorer (only "adx" or "both")
ADX_CLUSTER_URI = "https://<cluster>.<region>.kusto.windows.net"
# One ADX database per KQL database. ADX_DATABASE_PATTERN names it: {workspace} = workspace name,
# {database} = KQL database name, {id8} = first 8 characters of the KQL database item ID ({id} = all of it).
# The default is readable and unique across workspaces, e.g. "Sales_WS__SalesKQL__1b2c3d4e". The result is made
# safe for ADX (letters, digits, _ . - only). V2_6 doesn't need the names: each item records where its rows went.
ADX_DATABASE_PATTERN = "{workspace}__{database}__{id8}"
ADX_DATABASE_MAP = {}               # rarely needed: fixed name per KQL database {"name or item ID": "adx db"}
# True = V2_2 creates missing ADX databases (Azure Resource Manager, as the Kusto Copy service principal, which
# then is Admin on them). Needs Contributor on the ADX cluster. False = create them yourself first.
ADX_CREATE_DATABASES = True
ADX_SUBSCRIPTION_ID = ""            # "" = found from ADX_CLUSTER_URI (the app must see the cluster in Azure)
ADX_RESOURCE_GROUP = ""             # "" = found from ADX_CLUSTER_URI
ADX_SOFT_DELETE_PERIOD = ""         # retention of created databases; "" = unlimited (keep: Kusto Copy keeps the
                                    # original ingestion times, a short retention would drop old rows at once)
ADX_HOT_CACHE_PERIOD = "P1D"        # hot cache of created databases (backup data is rarely queried)
# Added as Viewer to each database V2_2 creates, so V2_6 can read it back. "me" = the person running this
# notebook; others as "aaduser=<upn>" or "aadgroup=<object id>;<tenant id>".
ADX_VIEWERS = ["me"]
ADX_CREATE_TABLES = True            # create missing destination tables (and add missing columns)

# Kusto Copy signs in as a service principal. Empty = the same app as the DevOps service principal (ADO_SP_*).
ADO_SP_TENANT_ID = ""
ADO_SP_CLIENT_ID = ""
ADO_SP_SECRET_NAME = ""
KC_TENANT_ID = ""
KC_CLIENT_ID = ""
KC_CLIENT_SECRET_NAME = ""

# The Kusto Copy Linux release downloaded from GitHub. It runs only if its SHA-256 matches.
KC_VERSION = "1.11.3.144"
KC_SHA256 = "dd546b3fcc2320085a0e2aff9af16ffe314df317fe95742628c161fcbc2a0b77"
KC_STAGING_FOLDER = "kusto-copy"    # in ADLS_CONTAINER. Keep it: it holds the progress used for the next run.
KC_COPY_MODE = "BackfillAndNew"     # first run copies history, later runs copy new arrivals only
KC_PARALLEL_JOBS = 2
KC_EXPORT_COUNT = 0
KC_TIMEOUT_HOURS = 8

## Helpers
Calls the Fabric, OneLake, ADLS, Eventhouse and Azure DevOps APIs with the signed-in user's token,
waiting on throttling (HTTP 429) and retrying transient errors.

In [ ]:
BUILD_ID = "3b4b3e2a00ba"
print("V2_2_Backup_Data build 3b4b3e2a00ba (compare with BUILD_IDS.txt of the package you meant to run)")
"""Credential redaction for diagnostic output, never for API or backup payloads."""
import re
from collections.abc import Mapping
from typing import TypeVar, overload
from urllib.parse import quote

REDACTED = "<redacted>"
if "_REDACTION_VALUES" not in globals():
    _REDACTION_VALUES: set[str] = set()

_REDACTION_KEYS = {
    "authorization", "accesstoken", "refreshtoken", "clientsecret", "serviceprincipalsecret",
    "password", "passwd", "accountkey", "apikey", "personalaccesstoken", "pat", "secret", "token",
    "connectionstring", "sastoken", "sharedaccesssignature", "sig",
}
_REDACTION_PATTERNS = (
    re.compile(r"\bgh[pousr]_[A-Za-z0-9]{20,}\b"),
    re.compile(r"\bgithub_pat_[A-Za-z0-9_]{40,}\b"),
    re.compile(r"\beyJ[A-Za-z0-9_-]{16,}\.[A-Za-z0-9_-]{16,}\.[A-Za-z0-9_-]{16,}\b"),
    re.compile(r"-----BEGIN (?:RSA |EC |OPENSSH )?PRIVATE KEY-----.*?"
               r"-----END (?:RSA |EC |OPENSSH )?PRIVATE KEY-----", re.DOTALL),
)
_REDACTION_AUTH = re.compile(r"\b(Bearer|Basic)([ \t]+)[A-Za-z0-9_+/=.-]{16,}", re.IGNORECASE)
_REDACTION_QUERY = re.compile(
    r"([?&](?:sig|access_token|refresh_token|client_secret|password|api_key)=)[^&\s\"'<>]+",
    re.IGNORECASE,
)
_REDACTION_ASSIGNMENT = re.compile(
    r"""(\b(?:client[_-]?secret|servicePrincipalSecret|access[_-]?token|refresh[_-]?token|
    password|passwd|account[_-]?key|api[_-]?key|personalAccessToken|sas[_-]?token|pat|secret|token)
    \b["']?\s*[:=]\s*["']?)([^\s"'&,;}\]<>]+)""",
    re.IGNORECASE | re.VERBOSE,
)


def register_sensitive_value(value: str) -> None:
    if len(value) >= 8:
        _REDACTION_VALUES.add(value)


def redact_sensitive_text(text: str) -> str:
    for value in sorted(_REDACTION_VALUES.copy(), key=len, reverse=True):
        text = text.replace(value, REDACTED).replace(quote(value, safe=""), REDACTED)
    for pattern in _REDACTION_PATTERNS:
        text = pattern.sub(REDACTED, text)
    text = _REDACTION_AUTH.sub(lambda m: m[1] + m[2] + REDACTED, text)
    text = _REDACTION_QUERY.sub(lambda m: m[1] + REDACTED, text)
    return _REDACTION_ASSIGNMENT.sub(lambda m: m[1] + REDACTED, text)


_Key = TypeVar("_Key")


@overload
def redact_sensitive_data(value: str) -> str: ...


@overload
def redact_sensitive_data(value: Mapping[_Key, object]) -> dict[_Key, object]: ...


@overload
def redact_sensitive_data(value: list[object]) -> list[object]: ...


@overload
def redact_sensitive_data(value: tuple[object, ...]) -> tuple[object, ...]: ...


@overload
def redact_sensitive_data(value: object) -> object: ...


def redact_sensitive_data(value: object) -> object:
    if isinstance(value, Mapping):
        out = {}
        for key, item in value.items():
            normalized = re.sub(r"[^a-z0-9]", "", str(key).lower())
            if normalized in _REDACTION_KEYS and item is not None and not (isinstance(item, str) and item == ""):
                out[key] = REDACTED
            else:
                out[key] = redact_sensitive_data(item)
        return out
    if isinstance(value, list):
        return [redact_sensitive_data(item) for item in value]
    if isinstance(value, tuple):
        return tuple(redact_sensitive_data(item) for item in value)
    return redact_sensitive_text(value) if isinstance(value, str) else value
import base64, datetime as dt, hashlib, json, re, threading, time, uuid
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import quote

import pandas as pd

pd.set_option("display.max_colwidth", None)   # never cut error messages or fixes in tables
import requests

try:
    notebookutils  # preloaded in Fabric notebooks
except NameError:
    import notebookutils

# ---------------------------------------------------------------- V2 settings -> backup switches
_mode = str(globals().get("KQL_BACKUP_MODE", "parquet")).strip().lower()
if _mode not in ("parquet", "adx", "both"):
    raise ValueError(f'KQL_BACKUP_MODE must be "parquet", "adx" or "both", not "{KQL_BACKUP_MODE}"')
KQL_BACKUP_MODE = _mode
EXPORT_KQL_TABLES = KQL_BACKUP_MODE in ("parquet", "both")
COPY_KQL_TO_ADX = KQL_BACKUP_MODE in ("adx", "both")
KC_TENANT_ID = str(globals().get("KC_TENANT_ID") or "").strip() or str(globals().get("ADO_SP_TENANT_ID") or "").strip() or "<tenant-id>"
KC_CLIENT_ID = str(globals().get("KC_CLIENT_ID") or "").strip() or str(globals().get("ADO_SP_CLIENT_ID") or "").strip() or "<app-client-id>"
KC_CLIENT_SECRET_NAME = (str(globals().get("KC_CLIENT_SECRET_NAME") or "").strip()
                         or str(globals().get("ADO_SP_SECRET_NAME") or "").strip() or "<secret-name>")
BACKUP_DEFINITIONS_TO_ADLS = True   # V2_3 checks the definition copy of every item
PUSH_DEFINITIONS_TO_DEVOPS = False  # V2: the workspace's own Git link (V2_1/V2_3) is the Git copy
ADO_ORGANIZATION = ADO_PROJECT = ADO_REPOSITORY = ADO_PAT_SECRET_NAME = ""
ADO_BRANCH, ADO_ROOT_FOLDER = "", ""
FREEZE_STARTED_UTC = str(globals().get("FREEZE_STARTED_UTC") or "").strip()

FABRIC_API = "https://api.fabric.microsoft.com/v1"
ONELAKE_DFS = "https://onelake.dfs.fabric.microsoft.com"
STORAGE_VERSION = "2023-11-03"
ADO_API = "api-version=7.1"
MAX_ATTEMPTS = 6
MAX_RETRY_WAIT_SECONDS = 15 * 60
LRO_TIMEOUT_SECONDS = 60 * 60
KQL_EXPORT_TIMEOUT_SECONDS = 4 * 60 * 60
CHUNK_BYTES = 8 * 1024 * 1024
ADO_PUSH_MAX_BYTES = 15 * 1024 * 1024
ADO_PUSH_MAX_CHANGES = 500
ZERO_GUID = "00000000-0000-0000-0000-000000000000"
GUID_RE = re.compile(r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$")
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
NO_DEFINITION_TYPES = {"SQLEndpoint"}           # no definition API: scripted with T-SQL instead
ONELAKE_DATA_TYPES = {"Lakehouse", "Warehouse"}
# Dataflow Gen2 staging items: Fabric creates them for Dataflows Gen2 (one Lakehouse and one Warehouse per workspace,
# named StagingLakehouseForDataflows_<timestamp> / StagingWarehouseForDataflows_<timestamp>) and creates them again
# when a Dataflow Gen2 runs. Not saved in Git, not backed up, not restored; V2_4 still deletes them (a Fabric item
# left behind blocks a tenant migration).
STAGING_RE = re.compile(r"^Staging(Lakehouse|Warehouse)ForDataflows(_\d+)?$", re.IGNORECASE)


def is_dataflow_staging(item):
    return bool(item.get("staging")) or (item.get("type") in ("Lakehouse", "Warehouse", "SQLEndpoint")
                                         and bool(STAGING_RE.match(str(item.get("displayName") or ""))))

RESULTS = []           # one row per item and step: the backup log
_lock = threading.Lock()
CALLS, RETRIES = Counter(), Counter()


def norm_id(value):
    text = str(value or "").strip().strip("{}").lower()
    return text if GUID_RE.match(text) else None


def safe_name(name, max_len=80):
    """A folder-safe version of a Fabric display name."""
    text = re.sub(r'[\\/:*?"<>|#%\x00-\x1f]', "_", str(name or "")).strip().strip(".")
    return (text[:max_len].rstrip(" .") or "unnamed")


def log(item, step, status, detail="", files=0, size=0, rows=None):
    detail = redact_sensitive_text(str(detail))
    with _lock:
        RESULTS.append({"Workspace": item.get("workspaceName", ""), "WorkspaceId": item.get("workspaceId", ""),
                        "Item": item.get("displayName", ""), "Type": item.get("type", ""),
                        "ItemId": item.get("id", ""), "Step": step, "Status": status, "Files": files,
                        "Bytes": size, "Rows": rows, "Detail": str(detail)})


def show(df, rows=200):
    try:
        display(df.head(rows))
    except NameError:
        print(df.head(rows).to_string())


def human(n):
    n = float(n or 0)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024 or unit == "TB":
            return f"{n:,.1f} {unit}"
        n /= 1024


class HttpError(Exception):
    def __init__(self, status, code, message, url):
        self.status, self.code, self.message, self.url = status, code or "", redact_sensitive_text(str(message or "")), url
        super().__init__(self.short())

    def short(self):
        head = f"HTTP {self.status} {self.code}".strip() if self.status else str(self.code)
        return f"{head}: {self.message}" if self.message else head


RAW_BODY_LIMIT = 4000   # only for non-JSON bodies (for example a whole HTML page); JSON error messages are kept whole


def _raw_text(text):
    text = re.sub(r"<[^>]+>", " ", text or "")
    text = re.sub(r"\s+", " ", text).strip()
    if len(text) <= RAW_BODY_LIMIT:
        return text
    return text[:RAW_BODY_LIMIT] + f" ... [{len(text) - RAW_BODY_LIMIT} more characters of the response body]"


def _error_details(resp):
    try:
        body = resp.json()
    except ValueError:
        return "", _raw_text(resp.text)
    if not isinstance(body, dict):
        return "", str(body)
    err = body.get("error") if isinstance(body.get("error"), dict) else body
    code = err.get("errorCode") or err.get("code") or err.get("typeKey") or err.get("error") or ""
    msg = err.get("message") or err.get("@message") or err.get("error_description") or ""
    return str(code), str(msg)


def _retry_after(resp, attempt):
    try:
        return max(1, int(float(resp.headers.get("Retry-After", ""))))
    except (TypeError, ValueError):
        return min(300, 10 * 2 ** (attempt - 1))


_local = threading.local()


def _session():
    if getattr(_local, "session", None) is None:
        _local.session = requests.Session()
    return _local.session


def http(method, url, audience=None, *, auth=None, headers=None, params=None, json_body=None, data=None,
         timeout=300, retry=True, label="other"):
    """One HTTP call with a fresh Entra token. Retries 408/429/5xx (honouring Retry-After) and network errors."""
    for attempt in range(1, MAX_ATTEMPTS + 1):
        hdrs = dict(headers or {})
        if auth:
            hdrs["Authorization"] = auth
        elif audience:
            hdrs["Authorization"] = "Bearer " + notebookutils.credentials.getToken(audience)
        if hdrs.get("Authorization"):
            register_sensitive_value(hdrs["Authorization"].split(" ", 1)[-1])
        try:
            resp = _session().request(method, url, params=params, json=json_body, data=data,
                                      headers=hdrs, timeout=timeout)
        except requests.RequestException as exc:
            if not retry or attempt == MAX_ATTEMPTS:
                raise HttpError(None, type(exc).__name__, str(exc), url)
            time.sleep(min(60, 5 * attempt))
            continue
        with _lock:
            CALLS[label] += 1
        if retry and (resp.status_code in (408, 429) or resp.status_code >= 500):
            wait = _retry_after(resp, attempt)
            if attempt < MAX_ATTEMPTS and wait <= MAX_RETRY_WAIT_SECONDS:
                with _lock:
                    RETRIES[label] += 1
                time.sleep(wait)
                continue
        if resp.status_code >= 400:
            raise HttpError(resp.status_code, *_error_details(resp), url)
        return resp


# ---------------------------------------------------------------- Fabric REST API
def _wait_operation(resp):
    """Follow a Fabric long-running operation (202 Accepted) to its result."""
    location = resp.headers.get("Location")
    op_id = resp.headers.get("x-ms-operation-id")
    if not location and op_id:
        location = f"{FABRIC_API}/operations/{op_id}"
    if not location:
        return {}
    wait = _retry_after(resp, 1) if resp.headers.get("Retry-After") else 2
    deadline = time.monotonic() + LRO_TIMEOUT_SECONDS
    while True:
        time.sleep(min(max(wait, 1), 30))
        state = http("GET", location, "pbi", label="fabric").json()
        status = str(state.get("status", "")).lower()
        if status == "succeeded":
            result = http("GET", location.rstrip("/") + "/result", "pbi", label="fabric")
            return result.json() if result.content else {}
        if status in ("failed", "cancelled", "canceled"):
            err = state.get("error") or {}
            raise HttpError(None, err.get("errorCode") or status, err.get("message"), location)
        if time.monotonic() > deadline:
            raise HttpError(None, "Timeout", "the long-running operation didn't finish", location)
        wait = 3


def fabric(method, path, body=None, params=None):
    resp = http(method, FABRIC_API + path, "pbi", json_body=body, params=params, label="fabric")
    if resp.status_code == 202:
        return _wait_operation(resp)
    return resp.json() if resp.content else {}


def fabric_pages(path, key="value", params=None):
    url, prm, seen = FABRIC_API + path, dict(params or {}), set()
    while url:
        body = http("GET", url, "pbi", params=prm or None, label="fabric").json()
        yield from body.get(key) or []
        if body.get("continuationUri"):
            url, prm = body["continuationUri"], {}
        elif body.get("continuationToken"):
            url, prm = FABRIC_API + path, {**(params or {}), "continuationToken": body["continuationToken"]}
        else:
            break
        marker = (url, json.dumps(prm, sort_keys=True))
        if marker in seen:
            break
        seen.add(marker)


# ---------------------------------------------------------------- ADLS Gen2 (destination)
ADLS_HEADERS = {"x-ms-version": STORAGE_VERSION}


def adls_url(path):
    return f"https://{ADLS_ACCOUNT}.dfs.core.windows.net/{ADLS_CONTAINER}/{quote(path.strip('/'), safe='/')}"


def adls_create(path):
    http("PUT", adls_url(path), "storage", headers=ADLS_HEADERS, params={"resource": "file"}, label="adls")


def adls_append(path, position, data):
    http("PATCH", adls_url(path), "storage", headers=ADLS_HEADERS, data=data,
         params={"action": "append", "position": str(position)}, label="adls")


def adls_flush(path, position):
    http("PATCH", adls_url(path), "storage", headers=ADLS_HEADERS, data=b"",
         params={"action": "flush", "position": str(position)}, label="adls")


def adls_write(path, data):
    """Write a whole (small) file, replacing any existing one."""
    if isinstance(data, str):
        data = data.encode("utf-8")
    adls_create(path)
    for pos in range(0, len(data), CHUNK_BYTES):
        adls_append(path, pos, data[pos:pos + CHUNK_BYTES])
    adls_flush(path, len(data))
    return len(data)


# ---------------------------------------------------------------- OneLake (source)
def onelake_list(workspace_id, item_id, folder):
    """Every file under <item>/<folder> as (relative path, size, last modified). Empty if the folder is missing."""
    params = {"resource": "filesystem", "recursive": "true", "directory": f"{item_id}/{folder}"}
    url = f"{ONELAKE_DFS}/{workspace_id}"
    while True:
        try:
            resp = http("GET", url, "storage", headers=ADLS_HEADERS, params=params, label="onelake")
        except HttpError as exc:
            if exc.status == 404:
                return
            raise
        for entry in resp.json().get("paths", []):
            if str(entry.get("isDirectory", "false")).lower() == "true":
                continue
            rel = entry["name"].split("/", 1)[1] if "/" in entry["name"] else entry["name"]
            yield rel, int(entry.get("contentLength") or 0), entry.get("lastModified", "")
        token = resp.headers.get("x-ms-continuation")
        if not token:
            return
        params = {**params, "continuation": token}


def onelake_read(workspace_id, item_id, rel_path, start, end):
    url = f"{ONELAKE_DFS}/{workspace_id}/{item_id}/{quote(rel_path, safe='/')}"
    resp = http("GET", url, "storage", headers={**ADLS_HEADERS, "Range": f"bytes={start}-{end}"}, label="onelake")
    return resp.content


def copy_onelake_file(workspace_id, item_id, rel_path, size, dest_path):
    """Copy one OneLake file to ADLS in chunks. Retries the whole file once if a chunk fails."""
    for attempt in (1, 2):
        try:
            adls_create(dest_path)
            pos = 0
            while pos < size:
                chunk = onelake_read(workspace_id, item_id, rel_path, pos, min(pos + CHUNK_BYTES, size) - 1)
                if not chunk:
                    raise HttpError(None, "ShortRead", f"no bytes returned at offset {pos}", rel_path)
                adls_append(dest_path, pos, chunk)
                pos += len(chunk)
            adls_flush(dest_path, pos)
            return pos
        except HttpError as exc:
            if attempt == 2 or exc.status == 404:
                raise
            time.sleep(5)


# ---------------------------------------------------------------- Kusto (Eventhouse)
def kusto_mgmt(query_uri, database, command, retry=True, auth=None, kind="mgmt"):
    """Run a Kusto control command (kind="mgmt") or query (kind="query"). auth overrides the runner's token."""
    resp = http("POST", query_uri.rstrip("/") + f"/v1/rest/{kind}", "kusto", auth=auth,
                json_body={"db": database, "csl": command}, label="kusto", retry=retry)
    tables = resp.json().get("Tables") or []
    if not tables:
        return []
    cols = [c["ColumnName"] for c in tables[0].get("Columns", [])]
    return [dict(zip(cols, row)) for row in tables[0].get("Rows", [])]


def kql_string(text):
    return "'" + str(text).replace("\\", "\\\\").replace("'", "\\'") + "'"


# ---------------------------------------------------------------- Kusto Copy service principal
_kc_tokens = {}


def kc_secret():
    if "secret" not in _kc_tokens:
        _kc_tokens["secret"] = notebookutils.credentials.getSecret(KEY_VAULT_URL, KC_CLIENT_SECRET_NAME)
        register_sensitive_value(_kc_tokens["secret"])
    return _kc_tokens["secret"]


def kc_auth(resource):
    """Authorization header for the Kusto Copy service principal (client credentials), cached until near expiry."""
    scope = resource.rstrip("/") + "/.default"
    token, expires = _kc_tokens.get(scope, (None, 0))
    if not token or time.time() > expires - 300:
        resp = http("POST", f"https://login.microsoftonline.com/{quote(KC_TENANT_ID)}/oauth2/v2.0/token",
                    data={"grant_type": "client_credentials", "client_id": KC_CLIENT_ID,
                          "client_secret": kc_secret(), "scope": scope}, label="entra")
        body = resp.json()
        token, expires = body["access_token"], time.time() + int(body.get("expires_in", 3600))
        register_sensitive_value(token)
        _kc_tokens[scope] = (token, expires)
    return "Bearer " + token


# ---------------------------------------------------------------- ADX databases (Azure Resource Manager)
ARM = "https://management.azure.com"
KUSTO_ARM_API = "2023-08-15"
ADX_ARM = {}            # cluster: id, name, location
ADX_DB_NAMES = {}       # lower-case name -> name, databases that exist on the cluster
ADX_CREATED = {}        # database -> "created" / "would create"


def adx_uri():
    return ADX_CLUSTER_URI.rstrip("/").lower()


def arm(method, path, body=None, api=KUSTO_ARM_API):
    url = path if path.startswith("http") else ARM + path
    if "api-version=" not in url:
        url += ("&" if "?" in url else "?") + "api-version=" + api
    return http(method, url, auth=kc_auth(ARM), json_body=body, label="azure")


def arm_list(path, api=KUSTO_ARM_API):
    out, url = [], path
    while url:
        body = arm("GET", url, api=api).json()
        out += body.get("value", [])
        url = body.get("nextLink")
    return out


def adx_cluster_arm():
    """The ADX cluster's Azure resource (found from ADX_CLUSTER_URI unless subscription and group are set)."""
    if ADX_ARM:
        return ADX_ARM
    name = adx_uri().split("//", 1)[-1].split(".")[0]
    if ADX_SUBSCRIPTION_ID and ADX_RESOURCE_GROUP:
        c = arm("GET", f"/subscriptions/{quote(ADX_SUBSCRIPTION_ID)}/resourceGroups/{quote(ADX_RESOURCE_GROUP)}"
                       f"/providers/Microsoft.Kusto/clusters/{quote(name)}").json()
    else:
        subs = [ADX_SUBSCRIPTION_ID] if ADX_SUBSCRIPTION_ID else \
            [x["subscriptionId"] for x in arm_list("/subscriptions", api="2022-12-01")]
        c = None
        for sub in subs:
            for cl in arm_list(f"/subscriptions/{quote(sub)}/providers/Microsoft.Kusto/clusters"):
                if str((cl.get("properties") or {}).get("uri", "")).rstrip("/").lower() == adx_uri():
                    c = cl
                    break
            if c:
                break
        if not c:
            raise RuntimeError(
                f"The Kusto Copy service principal can't see the ADX cluster {ADX_CLUSTER_URI} in Azure "
                f"(searched {len(subs)} subscription(s)). Give it Contributor on the cluster (Azure portal > the "
                "cluster > Access control (IAM)), or set ADX_SUBSCRIPTION_ID and ADX_RESOURCE_GROUP, or set "
                "ADX_CREATE_DATABASES = False and create the databases yourself.")
    ADX_ARM.update(id=c["id"], name=c["name"], location=c["location"])
    return ADX_ARM


def adx_can_create():
    """True if the service principal's Azure roles on the cluster allow creating databases."""
    import fnmatch
    perms = arm_list(adx_cluster_arm()["id"] + "/providers/Microsoft.Authorization/permissions", api="2022-04-01")
    want = "microsoft.kusto/clusters/databases/write"
    ok = lambda pats: any(fnmatch.fnmatch(want, str(x).lower()) for x in pats or [])
    return any(ok(p.get("actions")) and not ok(p.get("notActions")) for p in perms)


def adx_existing_databases():
    if not ADX_DB_NAMES:
        for d in arm_list(adx_cluster_arm()["id"] + "/databases"):
            n = d["name"].split("/")[-1]
            ADX_DB_NAMES[n.lower()] = n
    return ADX_DB_NAMES


def adx_safe_name(text):
    """Letters, digits, _ . - only; long names keep their start plus a hash, so they stay unique."""
    t = re.sub(r"_{3,}", "__", re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text))).strip("_.-") or "db"
    return t if len(t) <= 200 else t[:150] + "_" + hashlib.sha1(t.encode()).hexdigest()[:8]


def runner_principal():
    """Kusto principal of the person (or app) running the notebook, for ADX_VIEWERS = ["me"]."""
    try:
        part = notebookutils.credentials.getToken("pbi").split(".")[1]
        c = json.loads(base64.urlsafe_b64decode(part + "=" * (-len(part) % 4)))
    except Exception:
        return ""
    if c.get("idtyp") == "app" or ("appid" in c and "upn" not in c and "unique_name" not in c):
        return f"aadapp={c.get('appid', '')};{c.get('tid', '')}"
    upn = c.get("upn") or c.get("unique_name") or ""
    return f"aaduser={upn}" if upn else ""


def create_adx_database(name):
    """Create a ReadWrite database (the service principal becomes its Admin) and wait until it is ready."""
    props = {}
    if ADX_SOFT_DELETE_PERIOD:
        props["softDeletePeriod"] = ADX_SOFT_DELETE_PERIOD
    if ADX_HOT_CACHE_PERIOD:
        props["hotCachePeriod"] = ADX_HOT_CACHE_PERIOD
    path = adx_cluster_arm()["id"] + "/databases/" + quote(name, safe="")
    resp = arm("PUT", path, body={"kind": "ReadWrite", "location": adx_cluster_arm()["location"],
                                  "properties": props})
    op = resp.headers.get("Azure-AsyncOperation") or resp.headers.get("azure-asyncoperation")
    deadline = time.monotonic() + 900
    while time.monotonic() < deadline:
        if op:
            status = arm("GET", op).json().get("status", "")
        else:
            status = (arm("GET", path).json().get("properties") or {}).get("provisioningState", "")
        if status.lower() == "succeeded":
            break
        if status.lower() in ("failed", "canceled"):
            raise RuntimeError(f"creating ADX database {name} ended as {status}")
        time.sleep(10)
    else:
        raise RuntimeError(f"ADX database {name} wasn't ready after 15 minutes")
    ADX_DB_NAMES[name.lower()] = name
    ADX_CREATED[name] = "created"
    notes = []
    for v in ADX_VIEWERS:
        principal = runner_principal() if v == "me" else v
        if not principal:
            notes.append("couldn't tell who runs the notebook; add yourself as Viewer by hand")
            continue
        try:
            kusto_mgmt(ADX_CLUSTER_URI, name, f".add database [{kql_string(name)}] viewers ({kql_string(principal)}) "
                       "'Fabric migration backup (V2_2)'", auth=kc_auth(ADX_CLUSTER_URI))
            notes.append(f"Viewer: {principal}")
        except HttpError as exc:
            notes.append(f"Viewer {principal} not added ({exc.short()}); add it by hand")
    return notes


# ---------------------------------------------------------------- Azure DevOps
def git_blob_sha(data):
    return hashlib.sha1(b"blob %d\x00" % len(data) + data).hexdigest()


def ado_base():
    return (f"https://dev.azure.com/{quote(ADO_ORGANIZATION)}/{quote(ADO_PROJECT)}"
            f"/_apis/git/repositories/{quote(ADO_REPOSITORY)}")


_ado_auth = {}


def ado(method, path, params=None, body=None, retry=True):
    if "header" not in _ado_auth:
        pat = notebookutils.credentials.getSecret(KEY_VAULT_URL, ADO_PAT_SECRET_NAME)
        register_sensitive_value(pat)
        _ado_auth["header"] = "Basic " + base64.b64encode((":" + pat).encode()).decode()
    url = ado_base() + path + ("&" if "?" in path else "?") + ADO_API
    resp = http(method, url, auth=_ado_auth["header"], params=params, json_body=body, retry=retry, label="devops")
    if "json" not in resp.headers.get("Content-Type", "json"):
        raise HttpError(resp.status_code, "NotJson", "Azure DevOps didn't return JSON; check the organization, "
                        "project and token (an expired token is redirected to a sign-in page)", url)
    return resp.json() if resp.content else {}


print(f"Run ID: {RUN_ID}")

## 1. Find the items
Lists the items of the selected types in each workspace. KQL databases are added for the selected
Eventhouses. A workspace you can't open is logged and skipped.

In [ ]:
RESULTS.clear()
selected_types = set(ITEM_TYPES)
if INCLUDE_KQL_DATABASES and "Eventhouse" in selected_types:
    selected_types.add("KQLDatabase")
only_ids = {norm_id(x) for x in ONLY_ITEM_IDS} - {None}

workspace_ids = []
for raw in WORKSPACE_IDS:
    ws_id = norm_id(raw)
    if not ws_id:
        raise ValueError(f"Not a workspace ID (GUID): {raw!r}")
    if ws_id not in workspace_ids:
        workspace_ids.append(ws_id)
if not workspace_ids:
    raise ValueError("WORKSPACE_IDS is empty. Add the workspace IDs to back up in the settings cell.")

WORKSPACES, ITEMS, STAGING_SKIPPED = {}, [], []
git_ws_names = Counter()
for ws_id in workspace_ids:
    try:
        ws = fabric("GET", f"/workspaces/{ws_id}")
        all_items = list(fabric_pages(f"/workspaces/{ws_id}/items"))
    except HttpError as exc:
        print(f"  ! workspace {ws_id}: {exc.short()}")
        log({"workspaceId": ws_id}, "Inventory", "Failed",
            "No access to the workspace (needs Contributor or higher)" if exc.status in (401, 403) else exc.short())
        continue
    name = ws.get("displayName") or ws_id
    git_ws_names[safe_name(name)] += 1
    WORKSPACES[ws_id] = {"id": ws_id, "name": name, "adls": f"{safe_name(name)}__{ws_id}"}

    types_here = {i.get("type") for i in all_items}
    extra = {}
    if types_here & {"Lakehouse", "SQLEndpoint"}:
        for lh in fabric_pages(f"/workspaces/{ws_id}/lakehouses"):
            props = lh.get("properties") or {}
            extra[norm_id(lh["id"])] = props
            sql = props.get("sqlEndpointProperties") or {}
            if norm_id(sql.get("id")):
                extra[norm_id(sql["id"])] = {"lakehouseId": norm_id(lh["id"]),
                                             "connectionString": sql.get("connectionString")}
    if types_here & {"Eventhouse", "KQLDatabase"}:
        for db in fabric_pages(f"/workspaces/{ws_id}/kqlDatabases"):
            extra[norm_id(db["id"])] = db.get("properties") or {}

    picked = [i for i in all_items if i.get("type") in selected_types - {"KQLDatabase"}]
    picked_ids = {norm_id(i["id"]) for i in picked}
    for i in all_items:  # KQL databases follow their parent Eventhouse
        if i.get("type") == "KQLDatabase" and "KQLDatabase" in selected_types:
            parent = norm_id(extra.get(norm_id(i["id"]), {}).get("parentEventhouseItemId"))
            if parent in picked_ids:
                picked.append(i)
    for i in picked:
        iid = norm_id(i["id"])
        parent = norm_id(extra.get(iid, {}).get("parentEventhouseItemId"))
        if only_ids and iid not in only_ids and parent not in only_ids:
            continue
        if is_dataflow_staging(i):
            STAGING_SKIPPED.append(f"{name} / {i.get('displayName')} ({i['type']})")
            continue
        ITEMS.append({"id": iid, "displayName": i.get("displayName") or iid, "type": i["type"],
                      "description": i.get("description", ""), "folderId": i.get("folderId"),
                      "workspaceId": ws_id, "workspaceName": name, "properties": extra.get(iid, {})})

# Folder names: ADLS keeps the IDs; the Git copy uses Fabric Git integration's "<name>.<Type>" layout.
for ws in WORKSPACES.values():
    base = safe_name(ws["name"])
    ws["git"] = base if git_ws_names[base] == 1 else f"{base}_{ws['id'][:8]}"
seen = Counter((i["workspaceId"], safe_name(i["displayName"]).lower(), i["type"]) for i in ITEMS)
for i in ITEMS:
    base = safe_name(i["displayName"])
    unique = seen[(i["workspaceId"], base.lower(), i["type"])] == 1
    i["git"] = f"{base if unique else base + '_' + i['id'][:8]}.{i['type']}"
    i["adls"] = f"{ADLS_ROOT.strip('/')}/{RUN_ID}/{WORKSPACES[i['workspaceId']]['adls']}/{base}.{i['type']}__{i['id']}"

if not ITEMS:
    raise RuntimeError("No items of the selected types were found in the listed workspaces.")
plan = pd.DataFrame(ITEMS)
print(f"{len(ITEMS)} items in {len(WORKSPACES)} of {len(workspace_ids)} workspaces")
if STAGING_SKIPPED:
    print(f"Left out, Dataflow Gen2 staging items (Fabric recreates them; V2_4 still deletes them): "
          + ", ".join(STAGING_SKIPPED))
show(plan.pivot_table(index="workspaceName", columns="type", values="id", aggfunc="count", fill_value=0))

## 2. Check access to ADLS and Azure DevOps
Stops here with a clear message if the storage account or the repository can't be reached.

In [ ]:
RUN_PREFIX = f"{ADLS_ROOT.strip('/')}/{RUN_ID}"
placeholders = [n for n in ("ADLS_ACCOUNT", "ADLS_CONTAINER", "ADO_ORGANIZATION", "ADO_PROJECT", "ADO_REPOSITORY",
                            "KEY_VAULT_URL", "ADO_PAT_SECRET_NAME")
                if "<" in str(globals()[n]) and (n.startswith("ADLS") or PUSH_DEFINITIONS_TO_DEVOPS)]
KC_NEEDED = COPY_KQL_TO_ADX and any(i["type"] == "KQLDatabase" for i in ITEMS)
if KC_NEEDED:
    placeholders += [n for n in ("ADX_CLUSTER_URI", "KC_TENANT_ID", "KC_CLIENT_ID", "KC_CLIENT_SECRET_NAME",
                                 "KEY_VAULT_URL") if "<" in str(globals()[n]) and n not in placeholders]
if placeholders:
    raise ValueError("Fill in these settings first: " + ", ".join(placeholders) + (" (Kusto Copy: or ADO_SP_TENANT_ID / ADO_SP_CLIENT_ID / ADO_SP_SECRET_NAME)" if any(p.startswith("KC_") for p in placeholders) else ""))

STARTED_UTC = dt.datetime.now(dt.timezone.utc)
if FREEZE_STARTED_UTC:
    try:
        _freeze = dt.datetime.fromisoformat(FREEZE_STARTED_UTC.replace("Z", "+00:00"))
    except ValueError:
        raise ValueError(f'FREEZE_STARTED_UTC "{FREEZE_STARTED_UTC}" isn\'t a date and time like '
                         '"2026-11-16T10:00:00Z"') from None
    if _freeze.tzinfo is None:
        raise ValueError("FREEZE_STARTED_UTC needs the time zone: end it with Z (UTC)")
    if _freeze > STARTED_UTC:
        raise RuntimeError(f"FREEZE_STARTED_UTC ({FREEZE_STARTED_UTC}) is in the future: V2_3 needs a backup that "
                           "starts after the freeze. Run this after the freeze has started.")
    print(f"Change freeze started {FREEZE_STARTED_UTC}; this run starts {STARTED_UTC.isoformat()}")
else:
    print("WARNING: FREEZE_STARTED_UTC is empty. V2_3 will block this run until you set the freeze time there; "
          "fine for a test, not for the migration.")
print(f"Eventhouse data: KQL_BACKUP_MODE = \"{KQL_BACKUP_MODE}\"" + (
    "" if not COPY_KQL_TO_ADX else f" (ADX {ADX_CLUSTER_URI}, Kusto Copy as {KC_CLIENT_ID})"))
if DRY_RUN:
    print("DRY_RUN = True: nothing will be written to ADLS.")
else:
    try:
        adls_write(f"{RUN_PREFIX}/_run_started.json", json.dumps({
            "runId": RUN_ID, "startedUtc": STARTED_UTC.isoformat(), "freezeStartedUtc": FREEZE_STARTED_UTC,
            "kqlBackupMode": KQL_BACKUP_MODE, "adxCluster": ADX_CLUSTER_URI if COPY_KQL_TO_ADX else "",
            "workspaces": list(WORKSPACES.values()), "itemTypes": sorted(selected_types),
            "items": len(ITEMS)}, indent=2))
        print(f"ADLS OK: writing to abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT}.dfs.core.windows.net/{RUN_PREFIX}/")
    except HttpError as exc:
        raise RuntimeError(f"Can't write to ADLS ({exc.short()}). The person running the notebook needs "
                           "'Storage Blob Data Contributor' on the container, and the storage firewall must "
                           "allow this Fabric notebook.") from None

ADO_STATE = {}
if PUSH_DEFINITIONS_TO_DEVOPS and not DRY_RUN:
    try:
        repo = ado("GET", "")
        refs = ado("GET", "/refs", params={"filter": f"heads/{ADO_BRANCH}"}).get("value", [])
        ref = next((r for r in refs if r.get("name") == f"refs/heads/{ADO_BRANCH}"), None)
    except HttpError as exc:
        raise RuntimeError(f"Can't reach the Azure DevOps repository ({exc.short()}). Check the organization, "
                           "project, repository, Key Vault secret and the token's 'Code (Read & write)' scope."
                           ) from None
    if ref:
        ADO_STATE = {"head": ref["objectId"], "exists": True}
    else:
        default = repo.get("defaultBranch")
        base = None
        if default:
            refs = ado("GET", "/refs", params={"filter": default.replace("refs/", "", 1)}).get("value", [])
            base = next((r["objectId"] for r in refs if r.get("name") == default), None)
        ADO_STATE = {"head": base or "0" * 40, "exists": False}
    print(f"Azure DevOps OK: {ADO_REPOSITORY}, branch {ADO_BRANCH} "
          f"({'exists' if ADO_STATE['exists'] else 'will be created'})")

if KC_NEEDED:
    try:
        kc_auth(ADX_CLUSTER_URI)
    except Exception as exc:
        raise RuntimeError(f"The Kusto Copy service principal can't sign in ({exc}). Check KC_TENANT_ID, "
                           "KC_CLIENT_ID (empty = ADO_SP_TENANT_ID / ADO_SP_CLIENT_ID), the Key Vault secret, and "
                           "that you can read secrets in that vault.") from None
    print(f"Kusto Copy service principal OK: signed in for {ADX_CLUSTER_URI}")
    if ADX_CREATE_DATABASES:
        try:
            _c = adx_cluster_arm()
            _can = adx_can_create()
        except (HttpError, RuntimeError) as exc:
            raise RuntimeError(f"ADX_CREATE_DATABASES = True, but: {exc}") from None
        if not _can:
            raise RuntimeError(
                f"ADX_CREATE_DATABASES = True, but the Kusto Copy service principal ({KC_CLIENT_ID}) can't create "
                f"databases on {_c['name']}. Give it Contributor on the cluster (Azure portal > the ADX cluster > "
                "Access control (IAM) > Add role assignment), or create the databases yourself and set "
                "ADX_CREATE_DATABASES = False.")
        print(f"ADX cluster OK: {_c['name']} ({_c['location']}); the service principal can create missing databases "
              f"(names: {ADX_DATABASE_PATTERN})")

## 3. Item definitions
Gets each item's definition (Get Item Definition API) and keeps a copy in ADLS. `.platform` gets a
unique `logicalId` when Fabric returns none, which Git integration needs.

In [ ]:
DEFINITIONS = {}  # item id -> {part path: bytes}


def normalize_platform(item, parts):
    """Make sure .platform names the item and has a unique logicalId (Fabric Git needs one per item)."""
    try:
        platform = json.loads(parts[".platform"].decode("utf-8-sig")) if ".platform" in parts else {}
    except ValueError:
        platform = {}
    platform.setdefault("$schema", "https://developer.microsoft.com/json-schemas/fabric/gitIntegration/"
                                   "platformProperties/2.0.0/schema.json")
    meta = platform.setdefault("metadata", {})
    meta.setdefault("type", item["type"])
    meta.setdefault("displayName", item["displayName"])
    if item.get("description") and not meta.get("description"):
        meta["description"] = item["description"]
    config = platform.setdefault("config", {})
    config.setdefault("version", "2.0")
    if norm_id(config.get("logicalId")) in (None, ZERO_GUID):
        config["logicalId"] = str(uuid.uuid5(uuid.NAMESPACE_URL, "fabric-item/" + item["id"]))
    item["logicalId"] = config["logicalId"]
    parts[".platform"] = (json.dumps(platform, indent=2, ensure_ascii=False) + "\n").encode("utf-8")


def backup_definition(item):
    meta = {k: item.get(k) for k in ("id", "displayName", "type", "description", "folderId",
                                     "workspaceId", "workspaceName", "properties")}
    if not DRY_RUN:
        adls_write(f"{item['adls']}/item.json", json.dumps(meta, indent=2, ensure_ascii=False))
    if item["type"] in NO_DEFINITION_TYPES:
        log(item, "Definition", "Not applicable", "SQL analytics endpoints have no item definition; "
            "their views, functions and permissions are scripted in the SQL step")
        return
    try:
        result = fabric("POST", f"/workspaces/{item['workspaceId']}/items/{item['id']}/getDefinition")
    except HttpError as exc:
        if exc.status in (401, 403):
            status, detail = "Access denied", "Needs Contributor or higher on the workspace. " + exc.short()
        elif exc.code in ("OperationNotSupportedForItem", "UnsupportedItemType", "ItemDefinitionNotSupported"):
            status, detail = "Not supported", exc.short()
        else:
            status, detail = "Failed", exc.short()
        log(item, "Definition", status, detail)
        return
    parts, other = {}, []
    for part in (result.get("definition") or {}).get("parts", []):
        if part.get("payloadType", "InlineBase64") != "InlineBase64":
            other.append(part.get("path"))
            continue
        parts[part["path"].lstrip("/")] = base64.b64decode(part.get("payload") or "")
    if not parts:
        log(item, "Definition", "Failed", "The definition came back empty")
        return
    normalize_platform(item, parts)
    DEFINITIONS[item["id"]] = parts
    size = sum(len(v) for v in parts.values())
    if not DRY_RUN and BACKUP_DEFINITIONS_TO_ADLS:
        for path, data in parts.items():
            adls_write(f"{item['adls']}/definition/{path}", data)
    note = f"{len(parts)} parts" + (f"; skipped non-inline parts: {other}" if other else "")
    log(item, "Definition", "OK", note, files=len(parts), size=size)


with ThreadPoolExecutor(4) as pool:
    futures = {pool.submit(backup_definition, i): i for i in ITEMS}
    for n, fut in enumerate(as_completed(futures), 1):
        try:
            fut.result()
        except HttpError as exc:
            log(futures[fut], "Definition", "Failed", exc.short())
        if n % 25 == 0:
            print(f"  definitions: {n}/{len(ITEMS)}")

defs = pd.DataFrame([r for r in RESULTS if r["Step"] == "Definition"])
show(defs.pivot_table(index="Type", columns="Status", values="ItemId", aggfunc="count", fill_value=0))

## 5. Lakehouse and Warehouse data
Copies `Files/` and `Tables/` from OneLake to ADLS file by file. Delta log files are copied last.
For a consistent copy, run it when no loads are writing to these items.

In [ ]:
def shortcut_prefixes(item):
    if item["type"] != "Lakehouse":
        return []
    shortcuts = fabric_pages(f"/workspaces/{item['workspaceId']}/items/{item['id']}/shortcuts")
    return [f"{(s.get('path') or '').strip('/')}/{s.get('name', '')}".strip("/") for s in shortcuts]


def lakehouse_layout(item, files):
    """Schema-enabled Lakehouse: tables are Tables/<schema>/<table>; otherwise Tables/<table>. Recorded in
    lakehouse.json so V2_6 can check that the restored Lakehouse has the same layout before copying."""
    ds = (item.get("properties") or {}).get("defaultSchema") or ""
    tables = sorted({rel.split("/_delta_log/")[0] for rel, _, _ in files
                     if rel.startswith("Tables/") and "/_delta_log/" in rel})
    schemas = sorted({t.split("/")[1] for t in tables if t.count("/") >= 2}) if ds else []
    return {"schemaEnabled": bool(ds), "defaultSchema": ds, "schemas": schemas, "tables": tables}


def delta_check(item, files):
    """Read each table's Delta log in OneLake (JSON commits only, read-only) and make sure every data file it
    references is in the copy list. Warehouse data files don't always show up in a folder listing, so missing
    ones are added by the path in the log. Returns (extra files to copy, notes, problems)."""
    from urllib.parse import unquote
    listed = {rel for rel, _, _ in files}
    tables = sorted({rel.split("/_delta_log/")[0] for rel in listed if "/_delta_log/" in rel})
    extra, notes, problems, empty = [], [], [], []
    for table in tables:
        logs = sorted(rel for rel in listed if rel.startswith(table + "/_delta_log/")
                      and re.fullmatch(r"\d{20}\.json", rel.rsplit("/", 1)[-1]))
        if not logs or not logs[0].endswith("00000000000000000000.json"):
            notes.append(f"{table}: Delta log starts at a checkpoint, data files not cross-checked")
            continue
        active = {}
        for rel in logs:
            size = next(f[1] for f in files if f[0] == rel)
            text = onelake_read(item["workspaceId"], item["id"], rel, 0, size - 1).decode("utf-8") if size else ""
            for line in text.splitlines():
                try:
                    action = json.loads(line) if line.strip() else {}
                except ValueError:
                    continue
                if "add" in action:
                    active[action["add"]["path"]] = action["add"]
                elif "remove" in action:
                    active.pop(action["remove"]["path"], None)
        if not active:
            empty.append(table.split("/", 1)[-1])
            continue
        for path, add in active.items():
            if "://" in path:
                problems.append(f"{table}: data file outside the table folder ({path.split('?')[0]}), not copied")
                continue
            rel = f"{table}/{unquote(path)}"
            if rel not in listed:
                extra.append((rel, int(add.get("size") or 0), ""))
                listed.add(rel)
    if empty:
        notes.append(f"{len(empty)} table(s) with no data files in the published Delta log ({', '.join(empty[:10])}): "
                     "empty, or the Warehouse hasn't published its Delta log yet (check "
                     "SELECT name, DATA_LAKE_LOG_PUBLISHING_DESC FROM sys.databases; it must be AUTO)")
    if extra:
        notes.append(f"{len(extra)} data file(s) not in the folder listing were added from the Delta log")
    return extra, notes, problems


def layout_text(lay):
    if lay["schemaEnabled"]:
        return (f"schema-enabled (default schema {lay['defaultSchema']}): {len(lay['tables'])} table(s) in "
                f"{len(lay['schemas'])} schema(s) {', '.join(lay['schemas'])}".rstrip())
    return f"no schemas: {len(lay['tables'])} table(s)"


def backup_onelake_data(item):
    folders = []
    if item["type"] == "Lakehouse":
        folders = [f for f, on in (("Files", COPY_LAKEHOUSE_FILES), ("Tables", COPY_LAKEHOUSE_TABLES)) if on]
    elif item["type"] == "Warehouse" and COPY_WAREHOUSE_TABLES:
        folders = ["Tables"]
    if not folders:
        return
    ws_id, item_id = item["workspaceId"], item["id"]
    try:
        skip = shortcut_prefixes(item)
        files, skipped = [], 0
        for folder in folders:
            for rel, size, modified in onelake_list(ws_id, item_id, folder):
                if any(rel == p or rel.startswith(p + "/") for p in skip):
                    skipped += 1
                else:
                    files.append((rel, size, modified))
    except HttpError as exc:
        log(item, "Data", "Access denied" if exc.status in (401, 403) else "Failed", exc.short())
        return
    total = sum(f[1] for f in files)
    note = f"{len(skip)} shortcut(s) not copied ({skipped} files)" if skip else ""
    try:
        extra, dnotes, dproblems = delta_check(item, files)
    except (HttpError, StopIteration, UnicodeDecodeError) as exc:
        extra, dnotes, dproblems = [], [f"Delta log not cross-checked: {exc}"], []
    files += extra
    total = sum(f[1] for f in files)
    lay = lakehouse_layout(item, files) if item["type"] == "Lakehouse" else None
    if lay:
        note = (layout_text(lay) + (f". {note}" if note else "")).strip()
    note = ". ".join(x for x in [note] + dnotes + dproblems if x)
    if DRY_RUN:
        log(item, "Data", "Partial" if dproblems else "Dry run",
            f"would copy {len(files)} files, {human(total)}. {note}".strip(),
            files=len(files), size=total)
        return

    # Data files first, Delta logs last, so every log entry in the copy points at files already copied.
    first = [f for f in files if "/_delta_log/" not in "/" + f[0]]
    last = sorted(f for f in files if "/_delta_log/" in "/" + f[0])
    copied, copied_bytes, errors, manifest = 0, 0, [], ["path,size,lastModified,status"]
    started = time.monotonic()
    for batch in (first, last):
        with ThreadPoolExecutor(MAX_PARALLEL_FILES) as pool:
            futures = {pool.submit(copy_onelake_file, ws_id, item_id, rel, size, f"{item['adls']}/data/{rel}"):
                       (rel, size, modified) for rel, size, modified in batch}
            for fut in as_completed(futures):
                rel, size, modified = futures[fut]
                try:
                    copied_bytes += fut.result()
                    copied += 1
                    status = "copied"
                except HttpError as exc:
                    errors.append(f"{rel}: {exc.short()}")
                    status = "failed"
                manifest.append(",".join('"' + str(v).replace('"', '""') + '"' for v in (rel, size, modified, status)))
                if copied % 500 == 0 and copied:
                    print(f"    {item['displayName']}: {copied}/{len(files)} files, {human(copied_bytes)}, "
                          f"{time.monotonic() - started:,.0f} s")
    adls_write(f"{item['adls']}/data_manifest.csv", "\n".join(manifest) + "\n")
    if lay:
        adls_write(f"{item['adls']}/lakehouse.json", json.dumps(lay, indent=2))
    detail = f"{copied}/{len(files)} files, {human(copied_bytes)}. {note}".strip()
    if errors:
        detail += f" | {len(errors)} failed, first: {errors[0]}"
    errors += dproblems
    log(item, "Data", "OK" if not errors else ("Partial" if copied else "Failed"), detail,
        files=copied, size=copied_bytes)


data_items = [i for i in ITEMS if i["type"] in ONELAKE_DATA_TYPES]
for n, item in enumerate(data_items, 1):
    print(f"[{n}/{len(data_items)}] {item['type']} {item['workspaceName']} / {item['displayName']}")
    backup_onelake_data(item)
show(pd.DataFrame([r for r in RESULTS if r["Step"] == "Data"]).drop(columns=["WorkspaceId", "ItemId"], errors="ignore"))

## 6. Eventhouse data (KQL databases)
Saves the database schema script and exports every table to compressed Parquet with
`.export async`. The Eventhouse writes to ADLS directly, so the data doesn't pass through this notebook.

In [ ]:
TERMINAL_KQL_STATES = {"Completed", "Failed", "PartiallySucceeded", "Abandoned", "BadInput", "Canceled",
                       "Cancelled", "Throttled"}


def export_kql_table(uri, database, table, dest_folder):
    dest = (f"https://{ADLS_ACCOUNT}.blob.core.windows.net/{ADLS_CONTAINER}/"
            f"{quote(dest_folder, safe='/')}/{quote(safe_name(table), safe='')}")
    command = (f'.export async compressed to parquet (h@"{dest};impersonate") '
               f'with (namePrefix={kql_string(safe_name(table))}, parquetDatetimePrecision="microsecond") '
               f'<| [{kql_string(table)}]')
    op = kusto_mgmt(uri, database, command, retry=False)[0]["OperationId"]
    deadline = time.monotonic() + KQL_EXPORT_TIMEOUT_SECONDS
    while True:
        time.sleep(5)
        state = kusto_mgmt(uri, database, f".show operations {op}")[0]
        if state.get("State") in TERMINAL_KQL_STATES:
            break
        if time.monotonic() > deadline:
            raise HttpError(None, "Timeout", f"export operation {op} still running", uri)
    if state.get("State") != "Completed":
        raise HttpError(None, state.get("State"), state.get("Status") or f"operation {op}", uri)
    details = kusto_mgmt(uri, database, f".show operation {op} details")
    return len(details), sum(int(d.get("NumRecords") or 0) for d in details), \
        sum(int(d.get("SizeInBytes") or 0) for d in details)


def backup_kql_database(item):
    props = item["properties"] or {}
    if not props.get("queryServiceUri"):
        props = fabric("GET", f"/workspaces/{item['workspaceId']}/kqlDatabases/{item['id']}").get("properties", {})
    uri, database = props.get("queryServiceUri"), props.get("databaseName") or item["displayName"]
    if not uri:
        log(item, "KQL export", "Failed", "No query URI returned for this KQL database")
        return
    tables = kusto_mgmt(uri, database, ".show tables details")
    total_rows = sum(int(t.get("TotalRowCount") or 0) for t in tables)
    total_size = sum(int(t.get("TotalOriginalSize") or 0) for t in tables)
    if DRY_RUN:
        log(item, "KQL export", "Dry run", f"would export {len(tables)} tables, {total_rows:,} rows "
            f"(~{human(total_size)} uncompressed)", files=len(tables), size=total_size, rows=total_rows)
        return
    schema = kusto_mgmt(uri, database, ".show database schema as csl script")
    adls_write(f"{item['adls']}/database_schema.kql",
               "\n".join(str(r.get("DatabaseSchemaScript", "")) for r in schema) + "\n")
    files = rows = size = 0
    errors, mismatches = [], []
    for t in tables:
        name = t["TableName"]
        try:
            f, r, s = export_kql_table(uri, database, name, f"{item['adls']}/data")
            files, rows, size = files + f, rows + r, size + s
            if r != int(t.get("TotalRowCount") or 0):
                mismatches.append(f"{name}: {r:,} exported vs {int(t.get('TotalRowCount') or 0):,} at start")
        except HttpError as exc:
            errors.append(f"{name}: {exc.short()}")
        print(f"    {name}: done ({len(errors)} errors so far)")
    detail = f"{len(tables) - len(errors)}/{len(tables)} tables, {rows:,} rows, {files} Parquet files"
    if mismatches:
        detail += " | row counts changed during export (new ingestion?): " + "; ".join(mismatches[:5])
    if errors:
        detail += f" | {len(errors)} failed, first: {errors[0]}"
    log(item, "KQL export", "OK" if not errors else ("Partial" if len(errors) < len(tables) else "Failed"),
        detail, files=files, size=size, rows=rows)


def save_kql_schema_only(item):
    """KQL_BACKUP_MODE = "adx": no Parquet export, but keep the schema script in ADLS (V2_6 creates missing tables
    from it before copying the rows back from ADX)."""
    props = item["properties"] or {}
    if not props.get("queryServiceUri"):
        props = fabric("GET", f"/workspaces/{item['workspaceId']}/kqlDatabases/{item['id']}").get("properties", {})
    uri, database = props.get("queryServiceUri"), props.get("databaseName") or item["displayName"]
    if not uri:
        log(item, "KQL schema", "Failed", "No query URI returned for this KQL database")
        return
    if DRY_RUN:
        log(item, "KQL schema", "Dry run", "would save database_schema.kql")
        return
    schema = kusto_mgmt(uri, database, ".show database schema as csl script")
    adls_write(f"{item['adls']}/database_schema.kql",
               "\n".join(str(r.get("DatabaseSchemaScript", "")) for r in schema) + "\n")
    log(item, "KQL schema", "OK", "database_schema.kql saved (rows go to ADX: Kusto Copy step)")


kql_items = [i for i in ITEMS if i["type"] == "KQLDatabase"]
if not EXPORT_KQL_TABLES:
    print(f"Parquet export skipped (KQL_BACKUP_MODE = \"{KQL_BACKUP_MODE}\"); saving the schema scripts only.")
    for item in kql_items:
        try:
            save_kql_schema_only(item)
        except HttpError as exc:
            log(item, "KQL schema", "Access denied" if exc.status in (401, 403) else "Failed", exc.short())
for n, item in enumerate(kql_items if EXPORT_KQL_TABLES else [], 1):
    print(f"[{n}/{len(kql_items)}] KQL database {item['workspaceName']} / {item['displayName']}")
    try:
        backup_kql_database(item)
    except HttpError as exc:
        log(item, "KQL export", "Access denied" if exc.status in (401, 403) else "Failed", exc.short())

## 7. Copy Eventhouse tables to Azure Data Explorer (Kusto Copy)
Copies every table of the selected KQL databases into an Azure Data Explorer cluster with
[Kusto Copy](https://github.com/Azure/kusto-copy). The Eventhouse exports to the staging folder in
ADLS and ADX ingests from there, so the data doesn't pass through this notebook.

* **One ADX database per KQL database**, named by `ADX_DATABASE_PATTERN` (default
  `{workspace}__{database}__{id8}`, e.g. `Sales_WS__SalesKQL__1b2c3d4e`): readable, unique across workspaces,
  original table names kept. With `ADX_CREATE_DATABASES = True` (default) V2_2 creates missing databases through
  Azure Resource Manager as the service principal (which makes it Admin on them) and adds `ADX_VIEWERS` (you) as
  Viewer. The dry run lists the databases it would create. Every real run saves `adx_inventory.csv` (workspace,
  KQL database, ADX database, tables, rows) in the run folder; V2_6 doesn't need it, each item records its own
  destination.
* Each table is its own Kusto Copy job with a fixed staging folder
  (`<KC_STAGING_FOLDER>/<adx host>/<adx database>/<KQL database id>/<table>`). With
  `KC_COPY_MODE = "BackfillAndNew"` the first run copies the history and every later run copies
  only rows ingested since the previous run. **Don't delete or move these folders**: a new folder
  starts a new job and copies the history again, which duplicates rows in ADX.
* **The ADX database or a table was deleted (or emptied) after a copy?** The staging folder still says those
  rows were copied, so Kusto Copy would copy nothing. V2_2 checks this: for every ADX table that is missing or
  empty while its staging folder has progress, it resets that folder and copies the whole history again (no
  duplicates, the table is empty). The dry run lists these tables; the result says "old Kusto Copy progress reset".
* The notebook downloads Kusto Copy `KC_VERSION` (Linux) from GitHub releases, checks its SHA-256
  against `KC_SHA256`, and only then runs it. To move to a newer release, update both settings.
* Kusto Copy appends; it doesn't delete. Rows deleted or updated in the Eventhouse stay in ADX.
* A copy that hits `KC_TIMEOUT_HOURS`, or a notebook session that ends, isn't lost: the next run
  resumes the same job from its staging folder.

**Service principal permissions.** Kusto Copy signs in as an app, not as you: the app in `KC_CLIENT_ID`, or,
when that is empty, the DevOps service principal (`ADO_SP_CLIENT_ID`). Give **that app**:
* Fabric: member of each workspace (Viewer is enough to read the KQL databases).
* ADX: **Contributor** on the ADX cluster (Azure role, Access control (IAM)) so V2_2 can create the databases;
  it is then Admin on each database it creates. With `ADX_CREATE_DATABASES = False`: **Admin** on each destination
  database you created instead.
* Storage: **Storage Blob Data Contributor** on the container and **Storage Blob Delegator** on the
  storage account itself (account level) (Kusto Copy creates user-delegation SAS URLs for the Eventhouse and ADX).
* Network: the Eventhouse and the ADX ingestion service must reach the storage account, and this
  notebook must reach `github.com` (for the download) and the ADX cluster.


You (the person who later runs V2_6) need **Viewer** on each ADX database. Full checklist: README, *Prerequisites for the ADX copy*.

In [ ]:
import io, os, shutil, subprocess, tarfile, tempfile
from collections import deque

KC_STEP = "Kusto Copy"
ADX_URI = ADX_CLUSTER_URI.rstrip("/")
ADX_HOST = ADX_URI.split("//", 1)[-1].split("/")[0]
ANSI_RE = re.compile(r"\x1b\[[0-9;?]*[A-Za-z]")
NOTABLE_RE = re.compile(r"error|exception|fail|warn|denied|forbidden|unauthori", re.I)


def kql_endpoint(item):
    props = item["properties"] or {}
    if not props.get("queryServiceUri"):
        props = fabric("GET", f"/workspaces/{item['workspaceId']}/kqlDatabases/{item['id']}").get("properties", {})
    return props.get("queryServiceUri"), props.get("databaseName") or item["displayName"]


def adx_database_for(item, database):
    for key, value in ADX_DATABASE_MAP.items():
        if norm_id(key) == item["id"] or key == database:
            return value
    return adx_safe_name(ADX_DATABASE_PATTERN.format(database=database, workspace=item["workspaceName"],
                                                     id=item["id"], id8=item["id"][:8]))


def folder_part(text):
    """Stable, URL-safe folder name: readable prefix + hash of the exact name (so case and symbols can't collide)."""
    return re.sub(r"[^A-Za-z0-9_-]+", "_", text)[:60] + "_" + hashlib.sha1(text.encode()).hexdigest()[:8]


def staging_path(dest_db, item, table):
    return "/".join([KC_STAGING_FOLDER.strip("/"), ADX_HOST, folder_part(dest_db), item["id"], folder_part(table)])


def staging_uri(path):
    return f"https://{ADLS_ACCOUNT}.blob.core.windows.net/{ADLS_CONTAINER}/{path}"


def staging_has_progress(path):
    """True if Kusto Copy already left files (its progress) in this staging folder."""
    try:
        resp = http("GET", f"https://{ADLS_ACCOUNT}.dfs.core.windows.net/{ADLS_CONTAINER}", "storage",
                    headers=ADLS_HEADERS, label="adls", params={"resource": "filesystem", "recursive": "false",
                                                                "directory": path, "maxResults": "1"})
    except HttpError as exc:
        if exc.status == 404:
            return False
        raise
    return bool(resp.json().get("paths"))


def reset_staging(path):
    """Delete a staging folder (Kusto Copy's progress only; the rows are in the Eventhouse and copied again)."""
    params = {"recursive": "true"}
    while True:
        try:
            resp = http("DELETE", adls_url(path), "storage", headers=ADLS_HEADERS, params=params, label="adls")
        except HttpError as exc:
            if exc.status == 404:
                return
            raise
        token = resp.headers.get("x-ms-continuation")
        if not token:
            return
        params = {"recursive": "true", "continuation": token}


def yaml_str(value):
    return json.dumps(str(value), ensure_ascii=False)  # a JSON string is a valid YAML double-quoted scalar


def job_yaml(job):
    lines = ["activities:",
             f"  - activityName: {yaml_str(job['table'])}",
             "    source:",
             f"      clusterUri: {yaml_str(job['source_uri'])}",
             f"      databaseName: {yaml_str(job['source_db'])}",
             f"      tableName: {yaml_str(job['table'])}",
             "    destination:",
             f"      clusterUri: {yaml_str(ADX_URI)}",
             f"      databaseName: {yaml_str(job['dest_db'])}",
             f"      tableName: {yaml_str(job['table'])}",
             "stagingStorageDirectories:",
             f"  - {yaml_str(job['staging'])}",
             f"copyMode: {yaml_str(KC_COPY_MODE)}"]
    if KC_EXPORT_COUNT:
        lines.append(f"exportCount: {int(KC_EXPORT_COUNT)}")
    return "\n".join(lines) + "\n"


def redact(line):
    line = ANSI_RE.sub("", line).rstrip()
    line = re.sub(r"(https?://[^\s?\"']+)\?[^\s\"']*", r"\1?<redacted>", line)   # SAS query strings
    secret = _kc_tokens.get("secret")
    return redact_sensitive_text(line.replace(secret, "<redacted>") if secret else line)


def install_kusto_copy():
    url = f"https://github.com/Azure/kusto-copy/releases/download/{quote(KC_VERSION)}/kusto-copy-linux.tar.gz"
    data = http("GET", url, timeout=900, label="github").content
    digest = hashlib.sha256(data).hexdigest()
    if digest != KC_SHA256.strip().lower():
        raise RuntimeError(f"Kusto Copy download {KC_VERSION} has SHA-256 {digest}, expected {KC_SHA256}. "
                           "Not running it. Check KC_VERSION and KC_SHA256 against the GitHub release page.")
    folder = tempfile.mkdtemp(prefix="kusto-copy-")
    target = os.path.join(folder, "kc")
    with tarfile.open(fileobj=io.BytesIO(data), mode="r:gz") as tar:
        files = [m for m in tar.getmembers() if m.isfile()]
        exe = next((m for m in files if os.path.basename(m.name) in ("kc", "kusto-copy")), None)
        exe = exe or (files[0] if len(files) == 1 else None)
        if exe is None:
            raise RuntimeError("No Kusto Copy executable in the package: " + ", ".join(m.name for m in files))
        with tar.extractfile(exe) as src, open(target, "wb") as dst:   # only the executable, no path traversal
            shutil.copyfileobj(src, dst)
    os.chmod(target, 0o755)
    print(f"Kusto Copy {KC_VERSION} downloaded, SHA-256 verified: {target}")
    return target


def kc_environment():
    env = dict(os.environ)
    for name in ("AZURE_CLIENT_CERTIFICATE_PATH", "AZURE_FEDERATED_TOKEN_FILE", "AZURE_USERNAME", "AZURE_PASSWORD"):
        env.pop(name, None)
    env.update(AZURE_TENANT_ID=KC_TENANT_ID, AZURE_CLIENT_ID=KC_CLIENT_ID, AZURE_CLIENT_SECRET=kc_secret())
    env.setdefault("DOTNET_BUNDLE_EXTRACT_BASE_DIR", tempfile.gettempdir())
    # Kusto Copy asks for a named culture. With the ICU library (libicu) present, .NET uses it; without it, .NET
    # runs in invariant mode and must be allowed to map named cultures to the invariant one, or it stops with
    # CultureNotFoundException ("Only the invariant culture is supported").
    import glob
    has_icu = any(glob.glob(f"{d}/libicuuc.so*") for d in ("/usr/lib", "/usr/lib64", "/lib", "/lib64",
                                                           "/usr/lib/x86_64-linux-gnu", "/lib/x86_64-linux-gnu"))
    env["DOTNET_SYSTEM_GLOBALIZATION_INVARIANT"] = "0" if has_icu else "1"
    env["DOTNET_SYSTEM_GLOBALIZATION_PREDEFINED_CULTURES_ONLY"] = "false"
    return env


def run_kusto_copy(exe, job, env):
    """Run one table's job. Returns (exit code, timed out, log lines). The secret is passed only in the environment."""
    work = tempfile.mkdtemp(prefix="kc-job-")
    yaml_path = os.path.join(work, "job.yaml")
    with open(yaml_path, "w", encoding="utf-8") as f:
        f.write(job["yaml"])
    lines, last_print, timed_out = deque(maxlen=5000), 0.0, threading.Event()
    proc = subprocess.Popen([exe, "-y", yaml_path], cwd=work, env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, errors="replace", bufsize=1)

    def stop():
        timed_out.set()
        proc.kill()
    timer = threading.Timer(KC_TIMEOUT_HOURS * 3600, stop)
    timer.daemon = True
    timer.start()
    try:
        for raw in proc.stdout:
            line = redact(raw)
            if not line:
                continue
            lines.append(line)
            if NOTABLE_RE.search(line) or time.monotonic() - last_print > 600:   # errors, plus a heartbeat
                print(f"    [{job['table']}] {line}")
                last_print = time.monotonic()
        code = proc.wait()
    finally:
        timer.cancel()
        shutil.rmtree(work, ignore_errors=True)
    return code, timed_out.is_set(), list(lines)


def plan_database(item, seen):
    uri, database = kql_endpoint(item)
    if not uri:
        log(item, KC_STEP, "Failed", "No query URI returned for this KQL database")
        return []
    dest_db = adx_database_for(item, database)
    item["adxDatabase"] = dest_db
    missing = False
    if ADX_CREATE_DATABASES:
        try:
            existing = adx_existing_databases()
        except (HttpError, RuntimeError) as exc:
            log(item, KC_STEP, "Failed", f"Can't list the ADX databases in Azure: {exc}")
            return []
        if dest_db.lower() in existing:
            dest_db = item["adxDatabase"] = existing[dest_db.lower()]
        elif DRY_RUN:
            missing = True
            ADX_CREATED[dest_db] = "would create"
        else:
            print(f"  Creating ADX database {dest_db} on {ADX_HOST} ...")
            try:
                notes = create_adx_database(dest_db)
            except (HttpError, RuntimeError) as exc:
                log(item, KC_STEP, "Failed", f"Can't create ADX database '{dest_db}': {exc}. The Kusto Copy service "
                    "principal needs Contributor on the ADX cluster; or create the database yourself, make the "
                    "service principal Admin on it and set ADX_CREATE_DATABASES = False.")
                return []
            print(f"  Created ADX database {dest_db}" + (f" ({'; '.join(notes)})" if notes else ""))
    try:   # rows per destination table (an empty or missing table means a fresh copy, see stale_progress)
        dest_tables = {} if missing else \
            {r["TableName"]: int(r.get("TotalRowCount") or 0)
             for r in kusto_mgmt(ADX_URI, dest_db, ".show tables details", auth=kc_auth(ADX_URI))}
    except HttpError as exc:
        log(item, KC_STEP, "Access denied" if exc.status in (401, 403) else "Failed",
            f"ADX database '{dest_db}' on {ADX_HOST}: {exc.short()}. " + (
                "The service principal must be Admin on it (it is, on databases V2_2 created itself)."
                if ADX_CREATE_DATABASES else
                "Create the database first and make the service principal an Admin on it "
                "(or set ADX_CREATE_DATABASES = True)."))
        return []
    jobs, clashes = [], []
    for t in kusto_mgmt(uri, database, ".show tables details"):
        table = t["TableName"]
        key = (dest_db.lower(), table.lower())
        if key in seen and seen[key] != item["id"]:
            clashes.append(table)
            continue
        seen[key] = item["id"]
        path = staging_path(dest_db, item, table)
        job = {"item": item, "table": table, "source_uri": uri.rstrip("/"), "source_db": database,
               "dest_db": dest_db, "source_rows": int(t.get("TotalRowCount") or 0),
               "exists": table in dest_tables, "dest_rows": dest_tables.get(table, 0),
               "staging_path": path, "staging": staging_uri(path)}
        # Kusto Copy (BackfillAndNew) keeps in the staging folder what it already copied, and later runs copy only
        # newer rows. If the ADX table is empty or gone (database deleted and created again, table dropped or
        # cleared), that progress is stale: the run would copy nothing. Then the folder is reset and the whole
        # history copied again; no duplicates, the table is empty.
        job["stale_progress"] = job["dest_rows"] == 0 and staging_has_progress(path)
        job["yaml"] = job_yaml(job)
        jobs.append(job)
    if clashes:
        log(item, KC_STEP, "Failed", f"{len(clashes)} table(s) also come from another KQL database mapped to "
            f"ADX database '{dest_db}' and were skipped: {', '.join(clashes[:10])}. Map the databases to "
            "different ADX databases in ADX_DATABASE_MAP.")
    return jobs


def prepare_table(job):
    """Create the destination table (or add missing columns) from the source schema."""
    if not ADX_CREATE_TABLES:
        if not job["exists"]:
            raise HttpError(None, "Missing table", f"'{job['table']}' doesn't exist in ADX and ADX_CREATE_TABLES "
                            "is False", ADX_URI)
        return
    name = f"[{kql_string(job['table'])}]"
    row = kusto_mgmt(job["source_uri"], job["source_db"], f".show table {name} cslschema")[0]
    props = f" with (folder={kql_string(row['Folder'])})" if row.get("Folder") else ""
    kusto_mgmt(ADX_URI, job["dest_db"], f".create-merge table {name} ({row['Schema']}){props}",
               auth=kc_auth(ADX_URI))


def copy_table(exe, env, job):
    try:
        return _copy_table(exe, env, job)
    except Exception as exc:   # one table's failure must not stop the others
        return job, "Failed", redact(f"{type(exc).__name__}: {exc}"), None


def _copy_table(exe, env, job):
    item = job["item"]
    try:
        prepare_table(job)
    except HttpError as exc:
        return job, "Failed", f"create table: {exc.short()}", None
    note = ""
    if job["stale_progress"]:
        try:
            reset_staging(job["staging_path"])
        except HttpError as exc:
            return job, "Failed", (f"the ADX table is empty but Kusto Copy has progress from an earlier copy in "
                                   f"{job['staging_path']}, and resetting it failed: {exc.short()}. Delete that "
                                   "folder and run again."), None
        note = "; old Kusto Copy progress reset (the ADX table was empty), full history copied"
        print(f"    [{job['table']}] ADX table empty but staging has old progress: reset, copying the full history")
    started = time.monotonic()
    code, timed_out, lines = run_kusto_copy(exe, job, env)
    minutes = (time.monotonic() - started) / 60
    base = f"{item['adls']}/kusto_copy/{folder_part(job['table'])}"
    try:
        adls_write(base + ".yaml", job["yaml"])
        adls_write(base + ".log", "\n".join(lines) + "\n")
    except HttpError as exc:
        print(f"    [{job['table']}] log not saved: {exc.short()}")
    if timed_out:
        return job, "Partial", f"stopped after {KC_TIMEOUT_HOURS} h; the next run resumes it", None
    if code != 0:
        last = next((l for l in reversed(lines) if NOTABLE_RE.search(l)), lines[-1] if lines else "no output")
        return job, "Failed", f"Kusto Copy exit code {code}: {last}", None
    count = kusto_mgmt(ADX_URI, job["dest_db"], f"[{kql_string(job['table'])}] | count",
                       auth=kc_auth(ADX_URI), kind="query")
    rows = int(count[0].get("Count", 0)) if count else 0
    if rows < job["source_rows"]:
        return job, "Partial", (f"{rows:,} rows in ADX vs {job['source_rows']:,} in the source at the start "
                                "(check ADX retention, or rows still being ingested; Kusto Copy log: "
                                f"{base}.log){note}"), rows
    return job, "OK", f"{rows:,} rows in ADX ({minutes:.0f} min){note}", rows


def summarize(item, results):
    if not results:
        return
    dest = results[0][0]["dest_db"]
    ok = [r for r in results if r[1] == "OK"]
    bad = [r for r in results if r[1] != "OK"]
    rows = sum(r[3] or 0 for r in results)
    detail = f"{len(ok)}/{len(results)} tables copied to {ADX_HOST}/{dest}, {rows:,} rows in ADX"
    if bad:
        detail += f" | {len(bad)} not complete, first: {bad[0][0]['table']}: {bad[0][2]}"
    status = "OK" if not bad else ("Failed" if all(r[1] == "Failed" for r in results) else "Partial")
    log(item, KC_STEP, status, detail, files=len(results), rows=rows)
    inventory_row(item, dest, ADX_CREATED.get(dest, "existing"), len(results), rows, status)
    if not DRY_RUN:   # where the rows are, for V2_6 (restore from ADX)
        try:
            adls_write(f"{item['adls']}/kusto_copy/_destination.json", json.dumps({
                "cluster": ADX_CLUSTER_URI.rstrip("/"), "database": dest, "status": status,
                "tables": {r[0]["table"]: {"status": r[1], "rows": r[3]} for r in results}}, indent=2))
        except HttpError as exc:
            print(f"    destination record not saved: {exc.short()}")


ADX_INVENTORY = []


def inventory_row(item, dest, database_state, tables, rows, status):
    ADX_INVENTORY.append({"Workspace": item["workspaceName"], "WorkspaceId": item["workspaceId"],
                          "KQL database": item["displayName"], "KQL database item ID": item["id"],
                          "ADX cluster": ADX_URI, "ADX database": dest, "ADX database state": database_state,
                          "Tables": tables, "Rows": rows, "Status": status})


kql_items = [i for i in ITEMS if i["type"] == "KQLDatabase"]
if not COPY_KQL_TO_ADX:
    print(f"Kusto Copy to ADX skipped (KQL_BACKUP_MODE = \"{KQL_BACKUP_MODE}\").")
    kql_items = []
JOBS, seen = [], {}
for item in kql_items:
    print(f"Planning {item['workspaceName']} / {item['displayName']}")
    try:
        jobs = plan_database(item, seen)
    except HttpError as exc:
        log(item, KC_STEP, "Access denied" if exc.status in (401, 403) else "Failed", exc.short())
        continue
    if not jobs and not any(r["ItemId"] == item["id"] and r["Step"] == KC_STEP for r in RESULTS):
        log(item, KC_STEP, "Not applicable", "No tables in this KQL database")
    if DRY_RUN:
        if jobs:
            missing = sum(not j["exists"] for j in jobs)
            dest = jobs[0]["dest_db"]
            create = ADX_CREATED.get(dest) == "would create"
            log(item, KC_STEP, "Dry run", f"would copy {len(jobs)} tables, "
                f"{sum(j['source_rows'] for j in jobs):,} rows to {ADX_HOST}/{dest}"
                + (" (database to create)" if create else "") + f"; {missing} table(s) to create"
                + (f"; {stale} empty ADX table(s) with old Kusto Copy progress: reset and copied in full"
                   if (stale := sum(j["stale_progress"] for j in jobs)) else ""),
                files=len(jobs), rows=sum(j["source_rows"] for j in jobs))
            inventory_row(item, dest, "would create" if create else "existing", len(jobs),
                          sum(j["source_rows"] for j in jobs), "Dry run")
    else:
        JOBS += jobs

if JOBS:
    try:
        KC_EXE, KC_ENV = install_kusto_copy(), kc_environment()
    except (HttpError, RuntimeError, OSError, tarfile.TarError) as exc:
        for item in {j["item"]["id"]: j["item"] for j in JOBS}.values():
            log(item, KC_STEP, "Failed", f"Kusto Copy not installed: {exc}")
        JOBS = []
    by_item = defaultdict(list)
if JOBS:
    print(f"Copying {len(JOBS)} tables, {KC_PARALLEL_JOBS} at a time. Errors and a progress line every "
          "10 minutes per table are printed; the full logs go to ADLS (kusto_copy/ under each KQL database).")
    with ThreadPoolExecutor(max_workers=max(1, KC_PARALLEL_JOBS)) as pool:
        futures = [pool.submit(copy_table, KC_EXE, KC_ENV, j) for j in JOBS]
        for fut in as_completed(futures):
            job, status, detail, rows = fut.result()
            print(f"  {job['item']['displayName']} / {job['table']}: {status} - {detail}")
            by_item[job["item"]["id"]].append((job, status, detail, rows))
    for item in kql_items:
        summarize(item, by_item.get(item["id"], []))

if ADX_INVENTORY:
    print("\nWhere each KQL database goes in ADX (every run's list is saved as adx_inventory.csv in the run folder):")
    show(pd.DataFrame(ADX_INVENTORY).drop(columns=["WorkspaceId", "ADX cluster"]))
    if not DRY_RUN:
        try:
            adls_write(f"{RUN_PREFIX}/adx_inventory.csv", pd.DataFrame(ADX_INVENTORY).to_csv(index=False))
        except HttpError as exc:
            print(f"adx_inventory.csv not saved: {exc.short()}")

## 8. SQL analytics endpoints
SQL analytics endpoints have no definition API. This step saves the T-SQL of views, functions and
procedures, and the granted permissions. The tables themselves belong to the Lakehouse and are
copied in step 5.

In [ ]:
SQL_MODULES = """
SELECT s.name AS schema_name, o.name AS object_name, o.type_desc, m.definition
FROM sys.sql_modules AS m
JOIN sys.objects AS o ON o.object_id = m.object_id
JOIN sys.schemas AS s ON s.schema_id = o.schema_id
WHERE o.is_ms_shipped = 0
"""
SQL_PERMISSIONS = """
SELECT pr.name AS principal_name, pr.type_desc AS principal_type, pe.class_desc, pe.permission_name,
       pe.state_desc, OBJECT_SCHEMA_NAME(pe.major_id) AS schema_name, OBJECT_NAME(pe.major_id) AS object_name
FROM sys.database_permissions AS pe
JOIN sys.database_principals AS pr ON pr.principal_id = pe.grantee_principal_id
"""


def sql_connection(item):
    data_utils = getattr(notebookutils, "data", None)
    if data_utils is None or not hasattr(data_utils, "connect_to_artifact"):
        raise RuntimeError("notebookutils.data isn't available here; run this as a Python (not PySpark) notebook")
    attempts = [(item["id"], item["workspaceId"], item["type"])]
    if item["type"] == "SQLEndpoint" and item["properties"].get("lakehouseId"):
        attempts.append((item["properties"]["lakehouseId"], item["workspaceId"], "Lakehouse"))
    last = None
    for args in attempts:
        try:
            return data_utils.connect_to_artifact(*args)
        except Exception as exc:  # the utility raises several exception types
            last = exc
    raise RuntimeError(f"Couldn't connect: {str(last)}")


def query_frame(conn, text, columns):
    """The Fabric connector returns None, not an empty table, when a query has no rows."""
    df = conn.query(text)
    return pd.DataFrame(columns=columns) if df is None else df


def script_sql(item):
    if DRY_RUN:
        log(item, "SQL scripts", "Dry run", "would script views, functions, procedures and permissions")
        return
    try:
        conn = sql_connection(item)
        modules = query_frame(conn, SQL_MODULES, ["schema_name", "object_name", "type_desc", "definition"])
        permissions = query_frame(conn, SQL_PERMISSIONS, ["principal_name", "principal_type", "class_desc",
                                                          "permission_name", "state_desc", "schema_name",
                                                          "object_name"])
    except Exception as exc:
        log(item, "SQL scripts", "Failed", str(exc))
        return
    names = Counter()
    for row in modules.itertuples(index=False):
        base = f"{safe_name(row.schema_name)}/{safe_name(row.type_desc)}/{safe_name(row.object_name)}"
        names[base] += 1
        suffix = f"_{names[base]}" if names[base] > 1 else ""
        adls_write(f"{item['adls']}/sql/{base}{suffix}.sql", str(row.definition or "") + "\n")
    adls_write(f"{item['adls']}/sql/permissions.csv", permissions.to_csv(index=False))
    log(item, "SQL scripts", "OK", f"{len(modules)} objects, {len(permissions)} permission rows",
        files=len(modules) + 1)


for item in [i for i in ITEMS if i["type"] in SCRIPT_SQL_ITEM_TYPES]:
    script_sql(item)
show(pd.DataFrame([r for r in RESULTS if r["Step"] in ("SQL scripts", "KQL export")]))

## 9. Summary
Shows the outcome per step, lists everything that needs attention, and saves `backup_log.csv` to the run folder.

In [ ]:
log_df = pd.DataFrame(RESULTS, columns=["Workspace", "WorkspaceId", "Item", "Type", "ItemId", "Step", "Status",
                                        "Files", "Bytes", "Rows", "Detail"])
print("Backup result by step:")
show(log_df.pivot_table(index="Step", columns="Status", values="ItemId", aggfunc="count", fill_value=0))
problems = log_df[~log_df.Status.isin(["OK", "Unchanged", "Dry run", "Not applicable"])]
print(f"\n{len(problems)} rows need attention:")
show(problems.drop(columns=["WorkspaceId", "ItemId"]))
data_bytes = human(log_df[log_df.Step.isin(["Data", "KQL export"])].Bytes.sum())
print(f"\nData {'to copy (estimate, KQL uncompressed)' if DRY_RUN else 'copied'}: {data_bytes}")
print("API calls:", dict(CALLS), "| retried:", dict(RETRIES))

if not DRY_RUN:
    adls_write(f"{RUN_PREFIX}/backup_log.csv", log_df.to_csv(index=False))
    adls_write(f"{RUN_PREFIX}/_run_completed.json", json.dumps({
        "runId": RUN_ID, "finishedUtc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "items": len(ITEMS), "problems": len(problems), "kqlBackupMode": KQL_BACKUP_MODE,
        "freezeStartedUtc": FREEZE_STARTED_UTC}, indent=2))
    print(f"\nBackup folder: abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT}.dfs.core.windows.net/{RUN_PREFIX}/")
    print("\nNext: V2_3_Git_Snapshot with these settings" + (" (fix the rows above first: V2_3 blocks items whose "
          "backup isn't OK)" if len(problems) else "") + ":\n")
    print(f'ADLS_ACCOUNT = "{ADLS_ACCOUNT}"\nADLS_CONTAINER = "{ADLS_CONTAINER}"\nADLS_ROOT = "{ADLS_ROOT}"\n'
          f'SECONDARY_BACKUP_RUN_ID = "{RUN_ID}"\nFREEZE_STARTED_UTC = "{FREEZE_STARTED_UTC or "<set the freeze time>"}"')
else:
    print("\nDry run finished. Review the sizes above, set DRY_RUN = False and run all cells again.")

## Next step
Paste the printed settings into **V2_3_Git_Snapshot** and run it (dry run first). Restore after the move is done
by V2_5 (definitions from Git) and V2_6 (data from this run). If Azure DevOps is unavailable, V2_5 recreates the
items from ADLS instead (`RESTORE_SOURCE = "adls"`: the snapshot's raw copy, else this run's `definition/` folders).

### Troubleshooting
| Message | Meaning |
|---|---|
| `Access denied` on definitions or data | The runner isn't Contributor or higher on that workspace. |
| `Not supported` on a definition | Fabric has no definition API for that item (expected for SQLEndpoint). |
| `Can't write to ADLS` | Missing Storage Blob Data Contributor role, wrong container, or the storage firewall blocks Fabric. |
| KQL export fails with *access denied* or *forbidden* | The Eventhouse writes with the runner's identity (`impersonate`): check the runner's role on the container and the storage firewall. |
| Kusto Copy: *ADX database ... Create the database first* | The ADX database doesn't exist, or the service principal isn't an Admin on it. Map names in `ADX_DATABASE_MAP`. |
| Kusto Copy: *SHA-256 ... Not running it* | The download doesn't match `KC_SHA256`. Copy the release's SHA-256 from GitHub together with `KC_VERSION`. |
| Kusto Copy exit code with *unauthorized* or *forbidden* | Check the service principal: workspace member, ADX database Admin, Storage Blob Data Contributor on the container and Storage Blob Delegator on the account. |
| Kusto Copy: *configuration mismatch* | A staging folder's job changed (for example `KC_COPY_MODE`). Restore the setting, or change `KC_STAGING_FOLDER` to start over (which copies history again and duplicates rows already in ADX). |
| Kusto Copy: *rows in ADX vs ... in the source* | Fewer rows in ADX than the source had at the start. Check ADX retention: Kusto Copy keeps the original extent dates, so old rows can expire at once. |